### 01_extract_nodes.ipynb

This notebook extracts the data for each node/entity and saves the data to a `data/nodes/<node>.json`

In [2]:
import pandas as pd
import psycopg2
import os
import cenpy
#pip install cenpy@git+https://github.com/cenpy-devs/cenpy.git
#cd "/Applications/Python 3.13"./Install\ Certificates.command -- pip install --upgrade certifi
from dotenv import load_dotenv

C:\Users\Syed Haque\AppData\Roaming\Python\Python314\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


##### Connect to Data Sources

In [3]:
load_dotenv("../.env")

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

In [3]:
acs = cenpy.products.ACS()

In [4]:
conn = psycopg2.connect(
    host="awesome-hw.sdsc.edu",
    port=5432,
    dbname="nourish",
    user=DB_USER,
    password=DB_PASSWORD
)

def query_db(query):
    """Query the nourish database and return data as a `pd.Dataframe`"""
    try:
        with conn.cursor() as cursor:
            cursor.execute(query)
            columns = [desc[0] for desc in cursor.description]
            rows = cursor.fetchall()

        return pd.DataFrame(rows, columns=columns)
    except Exception as e:
        conn.rollback()
        raise e

query_db("SELECT 1")
print(f"Connection successful")

Connection successful


##### Utils

In [5]:
NODE_DIR = "../data/nodes/"

def save_df_to_json(df: pd.DataFrame, filename: str):
    """Save a pandas DataFrame to a JSON file."""
    with open(f"{NODE_DIR}/{filename}", "w") as f:
        df.to_json(f, index=False, orient="records", indent=2)

    print(f"Data saved to data/nodes/{filename}")

def clear_node_data_folder():
    """Clear all files in the data/nodes/ folder."""
    for filename in os.listdir(NODE_DIR):
        file_path = os.path.join(NODE_DIR, filename)
        try:
            if os.path.isfile(file_path):
                os.remove(file_path)
                print(f"Deleted file: {file_path}")
        except Exception as e:
            print(f"Error deleting file {file_path}: {e}")

##### Delete all node data files

In [6]:
clear_node_data_folder()

Deleted file: ../data/nodes/block_group.json
Deleted file: ../data/nodes/brand.json
Deleted file: ../data/nodes/business_location.json
Deleted file: ../data/nodes/city.json
Deleted file: ../data/nodes/community.json
Deleted file: ../data/nodes/county.json
Deleted file: ../data/nodes/sector.json
Deleted file: ../data/nodes/state.json
Deleted file: ../data/nodes/subsector.json
Deleted file: ../data/nodes/zipcode.json
Deleted file: ../data/nodes/zone_location.json
Deleted file: ../data/nodes/zone_type.json


##### Entity 1: `State`

In [7]:
query = """
    SELECT
    1 as id,
    'CA' as code,
    'California' as name
"""

state_df = query_db(query)
save_df_to_json(state_df, "state.json")
print(f"Rows: {state_df.shape[0]}, Columns: {state_df.shape[1]}")
state_df.head()

Data saved to data/nodes/state.json
Rows: 1, Columns: 3


,id,code,name
0,1,CA,California


##### Entity 2: `County`

In [8]:
query= """
SELECT
    id,
    county as name
FROM county_neighborhoods
"""

county_df = query_db(query)
save_df_to_json(county_df, "county.json")
print(f"Rows: {county_df.shape[0]}, Columns: {county_df.shape[1]}")
county_df.head()

Data saved to data/nodes/county.json
Rows: 58, Columns: 2


,id,name
0,1,Alameda
1,2,Alpine
2,3,Amador
3,4,Butte
4,5,Calaveras


##### Entity 3: `City`

In [9]:
query = """
SELECT
    c.id,
    c.city as name,
    ST_Transform(ST_Union(op.way), 4326) AS geom,
    ST_AsText(ST_Transform(ST_Union(op.way), 4326)) AS geom_wkt,
    ST_AsText(ST_Centroid(ST_Transform(ST_Union(op.way), 4326))) AS centroid_wkt
FROM
    city_neighborhoods c
LEFT JOIN osm_planet_socal_2025.osn_names ons
    ON c.city = ons.name
    AND ons.geom_type = 'polygon'
LEFT JOIN osm_planet_socal_2025.planet_osm_polygon op
    ON ons.osm_id = op.osm_id
    AND ons.geom_type = 'polygon'
WHERE
    op.osm_id < 0
    AND c.county = 'San Diego'
GROUP BY
    c.id,
    c.city
"""

city_df = query_db(query)
print(city_df.shape)
city_df.head(3)

(52, 5)


,id,name,geom,geom_wkt,centroid_wkt
0,8,Carlsbad,0103000020E6100000010000001D0300000EC40D53365A...,"POLYGON((-117.4095657 33.1325654995874,-117.40...",POINT(-117.31126249363 33.1196287846983)
1,9,Chula Vista,0103000020E610000002000000170600009710BDD6EF47...,"POLYGON((-117.1240136 32.6463658995912,-117.12...",POINT(-117.014430966873 32.62812821921)
2,10,Coronado,0103000020E6100000020000007A0100000DF3D4D97F4E...,"POLYGON((-117.2265534 32.6903901995905,-117.22...",POINT(-117.166004245592 32.6421665099776)


In [10]:
# #Add city attributes
cities_names_list = city_df['name'].unique()

attribute_variables = ['B01003_001E','B11001_001E']
cities_demographic_data = [['name', "total_population",'total_households']]

for city in cities_names_list:
  city_data =[]
  place_name = city + ', CA'
  # city_gdf = ox.geocode_to_gdf(place_name)
  # city_geometry = city_gdf.geometry.iloc[0]
  city_pop_gdf = acs.from_place(place_name, variables=attribute_variables)
  city_pop_tot = city_pop_gdf['B01003_001E'].sum()
  city_tot_household = city_pop_gdf['B11001_001E'].sum()
  city_data = [city, city_pop_tot, city_tot_household]
  cities_demographic_data.append(city_data)

cities_dem_df = pd.DataFrame(cities_demographic_data)
cities_dem_df.columns = cities_dem_df.iloc[0]
cities_dem_df = cities_dem_df[1:]
merged_city_df = pd.merge(city_df, cities_dem_df, on='name', how='inner')

save_df_to_json(merged_city_df, "city.json")
print(f"Rows: {merged_city_df.shape[0]}, Columns: {merged_city_df.shape[1]}")
merged_city_df.head()

Matched: Carlsbad, CA to Carlsbad city within layer Incorporated Places
Matched: Chula Vista, CA to Chula Vista city within layer Incorporated Places
Matched: Coronado, CA to Coronado city within layer Incorporated Places
Matched: Del Mar, CA to Del Mar city within layer Incorporated Places
Matched: El Cajon, CA to El Cajon city within layer Incorporated Places
Matched: Encinitas, CA to Encinitas city within layer Incorporated Places
Matched: Escondido, CA to Escondido city within layer Incorporated Places
Matched: Imperial Beach, CA to Imperial Beach city within layer Incorporated Places
Matched: La Mesa, CA to La Mesa city within layer Incorporated Places
Matched: Lemon Grove, CA to Lemon Grove city within layer Incorporated Places
Matched: National City, CA to National City city within layer Incorporated Places
Matched: Oceanside, CA to Oceanside city within layer Incorporated Places
Matched: Poway, CA to Poway city within layer Incorporated Places
Matched: San Diego, CA to San Dieg

C:\Users\Syed Haque\AppData\Roaming\Python\Python314\site-packages\cenpy\products.py:993: UserWarning: Cannot disambiguate placename Spring Valley. Picking the shortest, best matched placename, Spring Valley CDP, from Spring Valley CDP, Spring Valley CDP
  warn(


Matched: Spring Valley, CA to Spring Valley CDP within layer Census Designated Places
Matched: Valley Center, CA to Valley Center CDP within layer Census Designated Places
Matched: Winter Gardens, CA to Winter Gardens CDP within layer Census Designated Places
Data saved to data/nodes/city.json
Rows: 52, Columns: 7


,id,name,geom,geom_wkt,centroid_wkt,total_population,total_households
0,8,Carlsbad,0103000020E6100000010000001D0300000EC40D53365A...,"POLYGON((-117.4095657 33.1325654995874,-117.40...",POINT(-117.31126249363 33.1196287846983),34356.0,13245.0
1,9,Chula Vista,0103000020E610000002000000170600009710BDD6EF47...,"POLYGON((-117.1240136 32.6463658995912,-117.12...",POINT(-117.014430966873 32.62812821921),150885.0,44892.0
2,10,Coronado,0103000020E6100000020000007A0100000DF3D4D97F4E...,"POLYGON((-117.2265534 32.6903901995905,-117.22...",POINT(-117.166004245592 32.6421665099776),7363.0,2991.0
3,11,Del Mar,0103000020E610000001000000CD000000C7EDE1DC7051...,"POLYGON((-117.2725136 32.9801386995877,-117.27...",POINT(-117.262646302581 32.9632639479343),0.0,0.0
4,12,El Cajon,0103000020E61000000100000013090000DE2230D6B740...,"POLYGON((-117.0112205 32.8202865995889,-117.01...",POINT(-116.960489598889 32.8016821569438),49447.0,15381.0


##### Entity 4: `Community`

In [11]:
query = """
SELECT
    id,
    community as name
FROM community_neighborhoods
WHERE county = 'San Diego';
"""

community_df = query_db(query)
save_df_to_json(community_df, "community.json")
print(f"Rows: {community_df.shape[0]}, Columns: {community_df.shape[1]}")
community_df.head()

Data saved to data/nodes/community.json
Rows: 229, Columns: 2


,id,name
0,60,Midtown
1,55,Linda Vista
2,284,Eastlake Trails
3,285,Eastlake Vistas
4,283,Eastlake Land Swap


##### Entity 5: `Zipcode`

In [6]:
query = """
WITH sd_zipcodes AS (
    SELECT
        DISTINCT CAST(unnest(zipcodes) AS TEXT) as zipcode
    FROM city_neighborhoods
    WHERE county = 'San Diego'
)
SELECT
    t.zipcode,
    ST_Transform(ST_SetSRID(s.geom, 2230), 4326) AS geom,
    ST_AsText(ST_Transform(ST_SetSRID(s.geom, 2230), 4326)) AS geom_wkt,
    ST_AsText(ST_Centroid(ST_Transform(ST_SetSRID(s.geom, 2230), 4326))) AS centroid_wkt
FROM sd_zipcodes t
LEFT JOIN test_zipcodes s ON t.zipcode::int = s.zip::int;
"""

zipcode_df = query_db(query)
zipcode_df["id"] = zipcode_df.index + 1
zipcode_df = zipcode_df[["id"]+[col for col in zipcode_df.columns if col != "id"]]
save_df_to_json(zipcode_df, "zipcode.json")
print(f"Rows: {zipcode_df.shape[0]}, Columns: {zipcode_df.shape[1]}")
zipcode_df.head()

Data saved to data/nodes/zipcode.json
Rows: 107, Columns: 5


,id,zipcode,geom,geom_wkt,centroid_wkt
0,1,91901,0106000020E61000000100000001030000000400000087...,MULTIPOLYGON(((-116.719576934043 32.7296772458...,POINT(-116.69883366247 32.8094732720422)
1,2,91902,0106000020E610000001000000010300000007000000AA...,MULTIPOLYGON(((-117.021631240015 32.6887271248...,POINT(-117.013147370103 32.6723879949633)
2,3,91905,0106000020E610000002000000010300000005000000F1...,MULTIPOLYGON(((-116.358749029072 32.6010964598...,POINT(-116.306016727228 32.715857501762)
3,4,91906,0106000020E610000007000000010300000001000000E8...,MULTIPOLYGON(((-116.357667143203 32.6023411394...,POINT(-116.477962135902 32.6631950918073)
4,5,91910,0106000020E6100000020000000103000000010000000E...,MULTIPOLYGON(((-117.033257564554 32.6247476358...,POINT(-117.065681211833 32.636274324416)


##### Entity 6: `BusinessLocation`

In [ ]:
query = """
WITH sd_zipcodes AS (
    SELECT
        DISTINCT CAST(unnest(zipcodes) AS int) as zipcode
    FROM city_neighborhoods
    WHERE county = 'San Diego'
),
sd_businesses AS (
    SELECT
        b.*,
        -- Create the point geometry in WGS84 (Lat/Lon)
        ST_SetSRID(ST_MakePoint(longitude, latitude), 4326) AS geom
    FROM ca_business b
    JOIN sd_zipcodes z ON b.zip::int = z.zipcode::int
)
SELECT    
    b.id,
    b.name,
    b.url,
    b.address,
    b.city,
    b.zip,
    b.latitude,
    b.longitude,
    b.categories,
    b.avg_rating,
    -- Add the blockgroup column from the joined table
    scb.ctblockgroup AS blockgroup,
    ST_Transform(b.geom, 4326) AS geom,
    ST_AsText(ST_Transform(b.geom, 4326)) AS geom_wkt
FROM sd_businesses b
-- Perform the Spatial Join
LEFT JOIN sandag_layer_census_block_groups scb
    ON ST_Intersects(
        -- 1. Transform the business point from 4326 to 2230 (State Plane)
        ST_Transform(b.geom, 2230),
        -- 2. Compare against the census polygon 
        scb.geom
    );
"""

business_df = query_db(query)
business_df[business_df["zip"].isin(zipcode_df["zipcode"].values)]

print(business_df.shape)
business_df.head(3)

(45439, 13)


,id,name,url,address,city,zip,latitude,longitude,categories,avg_rating,blockgroup,geom,geom_wkt
0,62216,Casa Reveles Restaurant,https://www.google.com/maps/place//data=!4m2!3...,"Casa Reveles Restaurant, 28960 Lilac Rd, Valle...",Valley Center,92082,33.2365798,-117.05282589999999,[Mexican restaurant],4.3,191052.0,0101000020E61000006E38E27F61435DC076F2333F489E...,POINT(-117.0528259 33.2365798)
1,62285,Ochoa's Mexican Food,https://www.google.com/maps/place//data=!4m2!3...,"Ochoa's Mexican Food, 1051 E Main St, El Cajon...",El Cajon,92021,32.7943717,-116.9463024,[Mexican restaurant],4.1,157061.0,0101000020E61000001508F137903C5DC0182FCCF8AD65...,POINT(-116.9463024 32.7943717)
2,62287,Forever Green,https://www.google.com/maps/place//data=!4m2!3...,"Forever Green, 1551 S Mission Rd, Fallbrook, C...",Fallbrook,92028,33.3651155,-117.2500898,[Cannabis store],4.3,189062.0,0101000020E61000000B04A67801505DC0A0E1CD1ABCAE...,POINT(-117.2500898 33.3651155)


In [28]:
# Read Business LLM data
franchise_df = pd.read_json("../data/llm/sd_businesses.json")
print(franchise_df.shape)
franchise_df.head(3)

(39593, 17)


,id,business_name,url,address,city,zip_code,latitude,longitude,blockgroup,avg_rating,geom,franchise_type,is_franchise,has_valid_coordinates,category_original,category_sector,category_subsector
0,5,Internet Solutions For Less,https://www.google.com/maps/place//data=!4m2!3...,"Internet Solutions For Less, 733 Las Palmas Dr...",Vista,92081,33.186354,-117.250296,197021.0,5.0,POINT (-117.25029649999999 33.1863539),INDEPENDENT,0.0,True,Website designer,Other Services,Miscellaneous
1,24,Wetzel's Pretzels,https://www.google.com/maps/place//data=!4m2!3...,"Wetzel's Pretzels, 869 W. Harbor Drive, #C2-F,...",San Diego,92101,32.708565,-117.170277,54023.0,3.9,POINT (-117.17027739999999 32.708565199999995),FRANCHISE,1.0,True,Pretzel store,Other Services,Miscellaneous
2,48,Del Mar Golf Center - Pelly's Mini Golf,https://www.google.com/maps/place//data=!4m2!3...,"Del Mar Golf Center - Pelly's Mini Golf, 15555...",Del Mar,92014,32.976023,-117.253435,83241.0,4.4,POINT (-117.2534354 32.976022799999996),INDEPENDENT,0.0,True,Golf driving range,Entertainment & Recreation,Sports & Recreation


In [29]:
business_location_df = pd.merge(
    business_df,
    franchise_df[[
        'id',
        'has_valid_coordinates',
        'franchise_type', 
        'is_franchise',
        
    ]],
    on='id',
    how='inner'
)
print(business_location_df.shape)
business_location_df.head(3)

(39593, 16)


,id,name,url,address,city,zip,latitude,longitude,categories,avg_rating,blockgroup,geom,geom_wkt,has_valid_coordinates,franchise_type,is_franchise
0,62295,Cigar Grotto Inc.,https://www.google.com/maps/place//data=!4m2!3...,"Cigar Grotto Inc., 220 N Coast Hwy, Oceanside,...",Oceanside,92054,33.1966079,-117.3797924,"[Cigar shop, Tobacco shop]",4.8,184003.0,0101000020E61000003D51C8844E585DC04F519A722A99...,POINT(-117.3797924 33.1966079),True,INDEPENDENT,0.0
1,62313,A Gunalp Horologist Fine Watch & Clock Shop,https://www.google.com/maps/place//data=!4m2!3...,"A Gunalp Horologist Fine Watch & Clock Shop, 2...",San Diego,92110,32.750605,-117.192714,"[Watch repair service, Clock repair service]",4.6,65002.0,0101000020E6100000CEDE196D554C5DC0679B1BD31360...,POINT(-117.192714 32.750605),True,INDEPENDENT,0.0
2,62317,Surfsong Homeowners Association,https://www.google.com/maps/place//data=!4m2!3...,"Surfsong Homeowners Association, 233 S Helix A...",Solana Beach,92075,32.989122099999996,-117.27345899999999,[Homeowners' association],5,173071.0,0101000020E6100000FF722D5A80515DC01BA08F8D9B7E...,POINT(-117.273459 32.9891221),True,INDEPENDENT,0.0


In [30]:
save_df_to_json(business_location_df, "business_location.json")
print(f"Rows: {business_location_df.shape[0]}, Columns: {business_location_df.shape[1]}")
business_location_df.head()

Data saved to data/nodes/business_location.json
Rows: 39593, Columns: 16


,id,name,url,address,city,zip,latitude,longitude,categories,avg_rating,blockgroup,geom,geom_wkt,has_valid_coordinates,franchise_type,is_franchise
0,62295,Cigar Grotto Inc.,https://www.google.com/maps/place//data=!4m2!3...,"Cigar Grotto Inc., 220 N Coast Hwy, Oceanside,...",Oceanside,92054,33.1966079,-117.3797924,"[Cigar shop, Tobacco shop]",4.8,184003.0,0101000020E61000003D51C8844E585DC04F519A722A99...,POINT(-117.3797924 33.1966079),True,INDEPENDENT,0.0
1,62313,A Gunalp Horologist Fine Watch & Clock Shop,https://www.google.com/maps/place//data=!4m2!3...,"A Gunalp Horologist Fine Watch & Clock Shop, 2...",San Diego,92110,32.750605,-117.192714,"[Watch repair service, Clock repair service]",4.6,65002.0,0101000020E6100000CEDE196D554C5DC0679B1BD31360...,POINT(-117.192714 32.750605),True,INDEPENDENT,0.0
2,62317,Surfsong Homeowners Association,https://www.google.com/maps/place//data=!4m2!3...,"Surfsong Homeowners Association, 233 S Helix A...",Solana Beach,92075,32.989122099999996,-117.27345899999999,[Homeowners' association],5,173071.0,0101000020E6100000FF722D5A80515DC01BA08F8D9B7E...,POINT(-117.273459 32.9891221),True,INDEPENDENT,0.0
3,62321,Shirleys Kitchen,https://www.google.com/maps/place//data=!4m2!3...,"Shirleys Kitchen, 7118 University Ave, La Mesa...",La Mesa,91942,32.7548671,-117.0454863,"[Breakfast restaurant, Lunch restaurant]",4.6,147021.0,0101000020E6100000A0BA5E3FE9425DC0C6A9317C9F60...,POINT(-117.0454863 32.7548671),True,INDEPENDENT,0.0
4,62339,Beachworks,https://www.google.com/maps/place//data=!4m2!3...,"Beachworks, 4150 Mission Blvd UNIT 131, San Di...",San Diego,92109,32.791765999999996,-117.2543247,"[Swimwear store, Men's clothing store, Skate s...",4.6,79101.0,0101000020E610000029441BDB46505DC0D80A9A965865...,POINT(-117.2543247 32.791766),True,INDEPENDENT,0.0


##### Entity 7: `Brand`

In [17]:
brand_df = business_location_df[["name"]].drop_duplicates().reset_index(drop=True)
brand_df["id"] = brand_df.index + 1
brand_df = brand_df[["id"]+[col for col in brand_df.columns if col != "id"]]
save_df_to_json(brand_df, "brand.json")
print(f"Rows: {brand_df.shape[0]}, Columns: {brand_df.shape[1]}")
brand_df.head()

Data saved to data/nodes/brand.json
Rows: 32010, Columns: 2


,id,name
0,1,AutoZone Auto Parts
1,2,Cigar Grotto Inc.
2,3,A Gunalp Horologist Fine Watch & Clock Shop
3,4,Surfsong Homeowners Association
4,5,Shirleys Kitchen


##### Entity 8: `BlockGroup`

In [18]:
query = """
SELECT
    bd.std_geography_id as id,
    bd.std_geography_id AS geo_id,
    sbg.ctblockgroup,
    imp.countyfp,
    imp.tractce,
    imp.blkgrpce,
    imp.ogc_fid,
    imp.statefp,


    -- INCOME
    imp.AVGDI_CY as average_income,
    imp.MEDDI_CY as median_income,
    imp.di0_cy    AS income_under_15000,
    imp.di15_cy   AS income_15000_24999,
    imp.di25_cy   AS income_25000_34999,
    imp.di35_cy   AS income_35000_49999,
    imp.di50_cy   AS income_50000_74999,
    imp.di75_cy   AS income_75000_99999,
    imp.di100_cy  AS income_100000_149999,
    imp.di150_cy  AS income_150000_199999,
    imp.di200_cy  AS income_200000_plus,

    -- POPULATION
    imp.TOTPOP_CY as total_population,

    -- AGE
    (imp.male0 + imp.male5 + imp.fem0 + imp.fem5) AS population_0_9,
    (imp.male10 + imp.male15 + imp.fem10 + imp.fem15) AS population_10_19,
    (imp.male20 + imp.male25 + imp.fem20 + imp.fem25) AS population_20_29,
    (imp.male30 + imp.male35 + imp.fem30 + imp.fem35) AS population_30_39,
    (imp.male40 + imp.male45 + imp.fem40 + imp.fem45) AS population_40_49,
    (imp.male50 + imp.male55 + imp.fem50 + imp.fem55) AS population_50_59,
    (imp.male60 + imp.male65 + imp.fem60 + imp.fem65) AS population_60_69,
    (imp.male70 + imp.male75 + imp.fem70 + imp.fem75) AS population_70_79,
    (imp.male80 + imp.male85 + imp.fem80 + imp.fem85) AS population_80_plus,

    -- MALE
    (imp.male0  + imp.male5)  AS male_population_0_9,
    (imp.male10 + imp.male15) AS male_population_10_19,
    (imp.male20 + imp.male25) AS male_population_20_29,
    (imp.male30 + imp.male35) AS male_population_30_39,
    (imp.male40 + imp.male45) AS male_population_40_49,
    (imp.male50 + imp.male55) AS male_population_50_59,
    (imp.male60 + imp.male65) AS male_population_60_69,
    (imp.male70 + imp.male75) AS male_population_70_79,
    (imp.male80 + imp.male85) AS male_population_80_plus,
    (imp.male0  + imp.male5  +
     imp.male10 + imp.male15 +
     imp.male20 + imp.male25 +
     imp.male30 + imp.male35 +
     imp.male40 + imp.male45 +
     imp.male50 + imp.male55 +
     imp.male60 + imp.male65 +
     imp.male70 + imp.male75 +
     imp.male80 + imp.male85) AS male_population_total,

    -- FEMALE
    (imp.fem0  + imp.fem5)  AS female_population_0_9,
    (imp.fem10 + imp.fem15) AS female_population_10_19,
    (imp.fem20 + imp.fem25) AS female_population_20_29,
    (imp.fem30 + imp.fem35) AS female_population_30_39,
    (imp.fem40 + imp.fem45) AS female_population_40_49,
    (imp.fem50 + imp.fem55) AS female_population_50_59,
    (imp.fem60 + imp.fem65) AS female_population_60_69,
    (imp.fem70 + imp.fem75) AS female_population_70_79,
    (imp.fem80 + imp.fem85) AS female_population_80_plus,
    (imp.fem0  + imp.fem5  +
     imp.fem10 + imp.fem15 +
     imp.fem20 + imp.fem25 +
     imp.fem30 + imp.fem35 +
     imp.fem40 + imp.fem45 +
     imp.fem50 + imp.fem55 +
     imp.fem60 + imp.fem65 +
     imp.fem70 + imp.fem75 +
     imp.fem80 + imp.fem85) AS female_population_total,

    -- BUSINESS COUNTS
    bd.s01_bus  AS businesses_total_sic,
    bd.s02_bus  AS businesses_agriculture_mining_sic,
    bd.s08_bus  AS businesses_wholesale_trade_sic,
    bd.s09_bus  AS businesses_retail_trade_sic,
    bd.s12_bus  AS businesses_food_stores_sic,
    bd.s13_bus  AS businesses_auto_gas_sic,
    bd.s16_bus  AS businesses_eating_drinking_sic,
    bd.s17_bus  AS businesses_misc_retail_sic,
    bd.s18_bus  AS businesses_finance_insurance_realestate_sic,
    bd.s19_bus  AS businesses_banks_sic,
    bd.s20_bus  AS businesses_securities_sic,
    bd.s21_bus  AS businesses_insurance_sic,
    bd.s22_bus  AS businesses_real_estate_sic,
    bd.s23_bus  AS businesses_services_sic,
    bd.s24_bus  AS businesses_hotels_sic,
    bd.s25_bus  AS businesses_auto_services_sic,
    bd.s26_bus  AS businesses_amusements_sic,
    bd.s27_bus  AS businesses_health_services_sic,
    bd.s28_bus  AS businesses_legal_services_sic,
    bd.s29_bus  AS businesses_education_sic,
    bd.s30_bus  AS businesses_other_services_sic,

    bd.n01_bus  AS businesses_total_naics,
    bd.n02_bus  AS businesses_agriculture_naics,
    bd.n07_bus  AS businesses_wholesale_trade_naics,
    bd.n08_bus  AS businesses_retail_trade_naics,
    bd.n12_bus  AS businesses_building_materials_naics,
    bd.n13_bus  AS businesses_food_beverage_stores_naics,
    bd.n14_bus  AS businesses_health_personal_care_naics,
    bd.n15_bus  AS businesses_gas_stations_naics,
    bd.n27_bus  AS businesses_real_estate_leasing_naics,
    bd.n35_bus  AS businesses_accommodation_food_naics,
    bd.n36_bus  AS businesses_accommodation_naics,
    bd.n37_bus  AS businesses_food_services_naics,

    -- BUSINESS SALES
    bd.s01_sales AS sales_total_sic,
    bd.s02_sales AS sales_agriculture_mining_sic,
    bd.s08_sales AS sales_wholesale_trade_sic,
    bd.s09_sales AS sales_retail_trade_sic,
    bd.s12_sales AS sales_food_stores_sic,
    bd.s13_sales AS sales_auto_gas_sic,
    bd.s16_sales AS sales_eating_drinking_sic,
    bd.s17_sales AS sales_misc_retail_sic,
    bd.s18_sales AS sales_finance_insurance_realestate_sic,
    bd.s19_sales AS sales_banks_sic,
    bd.s20_sales AS sales_securities_brokers_sic,
    bd.s21_sales AS sales_insurance_sic,
    bd.s22_sales AS sales_real_estate_sic,
    bd.s23_sales AS sales_services_sic,
    bd.s24_sales AS sales_hotels_sic,
    bd.s25_sales AS sales_auto_services_sic,
    bd.s26_sales AS sales_amusement_sic,
    bd.s27_sales AS sales_health_services_sic,
    bd.s28_sales AS sales_legal_services_sic,
    bd.s29_sales AS sales_education_sic,
    bd.s30_sales AS sales_other_services_sic,

    bd.n01_sales AS sales_total_naics,
    bd.n02_sales AS sales_agriculture_naics,
    bd.n07_sales AS sales_wholesale_trade_naics,
    bd.n08_sales AS sales_retail_trade_naics,
    bd.n12_sales AS sales_building_materials_naics,
    bd.n13_sales AS sales_food_beverage_stores_naics,
    bd.n14_sales AS sales_health_personal_care_naics,
    bd.n15_sales AS sales_gas_stations_naics,
    bd.n27_sales AS sales_real_estate_leasing_naics,
    bd.n35_sales AS sales_accommodation_food_naics,
    bd.n36_sales AS sales_accommodation_naics,
    bd.n37_sales AS sales_food_services_naics,


    -- CONSUMER SPENDING
    cs.x1002_x::int                  AS cs_food,
    cs.x1003_x::int                  AS cs_food_at_home,
    cs.x1130_x::int                  AS cs_food_away_from_home,
    
    (cs.x1147_x::int
    + cs.x1148_x::int
    + cs.x1149_x::int
    + cs.x1150_x::int
    + cs.x1151_x::int)              AS cs_breakfast,

    (cs.x1132_x::int
    + cs.x1133_x::int
    + cs.x1134_x::int
    + cs.x1135_x::int
    + cs.x1136_x::int)              AS cs_lunch,

    (cs.x1137_x::int
    + cs.x1138_x::int
    + cs.x1139_x::int
    + cs.x1140_x::int
    + cs.x1141_x::int)              AS cs_dinner,

    (cs.x1142_x::int
    + cs.x1143_x::int
    + cs.x1144_x::int
    + cs.x1145_x::int
    + cs.x1146_x::int
    + cs.x1119_x::int
    + cs.x1120_x::int
    + cs.x1121_x::int
    + cs.x1126_x::int
    + cs.x1127_x::int
    + cs.x1128_x::int
    + cs.x1090_x::int
    + cs.x1106_x::int
    + cs.x7017_x::int
    + cs.x7018_x::int
    + cs.x7019_x::int
    + cs.x7020_x::int)               AS cs_snacks_beverages,

    (cs.x1133_x::int
    + cs.x1138_x::int
    + cs.x1143_x::int
    + cs.x1148_x::int)              AS cs_fast_food,


    (cs.x1134_x::int
    + cs.x1139_x::int
    + cs.x1144_x::int
    + cs.x1149_x::int)               AS cs_restuarant,

    (cs.x1135_x::int
    + cs.x1140_x::int
    + cs.x1145_x::int
    + cs.x1150_x::int
    + cs.x1152_x::int
    + cs.x1154_x::int)               AS cs_school_food,

    cs.x7018_x::int                  AS cs_alcohol,
    (cs.x1122_x::int
    + cs.x1123_x::int
    + cs.x1124_x::int
    + cs.x1125_x::int)               AS cs_coffee_tea,


    cs.x15001_x::int                 AS cs_retail_goods,

    (cs.x3002_x::int
    + cs.x3004_x::int
    + cs.x3005_x::int
    + cs.x3006_x::int
    + cs.x3036_x::int
    + cs.x3038_x::int)               AS cs_housing,
    cs.x9036_x::int                  AS cs_pet_food,


    -- GEOMETRY
    ST_Transform(sbg.geom, 4326) AS geom,
    ST_AsText(ST_Transform(geom, 4326)) AS geom_wkt,
    ST_AsText(ST_Centroid(ST_Transform(geom, 4326))) AS centroid_wkt

FROM sandag_layer_census_block_groups sbg
LEFT JOIN bgs_sd_imp imp
    ON CAST(sbg.ctblockgroup AS TEXT) = CAST(CONCAT(LTRIM(imp.tractce, '0'), imp.blkgrpce) AS TEXT)
LEFT JOIN esri_business_data bd
    ON TRIM(LEADING '0' FROM SUBSTR(CAST(bd.std_geography_id AS TEXT), 5)) = CAST(sbg.ctblockgroup AS TEXT)
LEFT JOIN esri_consumer_spending_cols cs
    ON TRIM(LEADING '0' FROM SUBSTR(CAST(cs.std_geography_id AS TEXT), 5)) = CAST(sbg.ctblockgroup AS TEXT)
WHERE
    imp.countyfp = '073';
"""

block_group_df = query_db(query)
save_df_to_json(block_group_df, "block_group.json")
print(f"Rows: {block_group_df.shape[0]}, Columns: {block_group_df.shape[1]}")
block_group_df.head() 

Data saved to data/nodes/block_group.json
Rows: 2057, Columns: 133


,id,geo_id,ctblockgroup,countyfp,tractce,blkgrpce,ogc_fid,statefp,average_income,median_income,...,cs_restuarant,cs_school_food,cs_alcohol,cs_coffee_tea,cs_retail_goods,cs_housing,cs_pet_food,geom,geom_wkt,centroid_wkt
0,60730010001,60730010001,10001,073,001000,1,497,06,84570.0,65007.0,...,1679956.0,140611.0,78255.0,236626.0,18069548.0,39402526.0,193892.0,0106000020E61000000100000001030000000100000081...,MULTIPOLYGON(((-117.139552001244 32.7672929995...,POINT(-117.141339781151 32.763522520968)
1,60730100011,60730100011,100011,073,010001,1,932,06,88556.0,74198.0,...,791262.0,77156.0,38808.0,111285.0,9388369.0,19533948.0,99243.0,0106000020E6100000010000000103000000010000005E...,MULTIPOLYGON(((-117.050494000916 32.5838399992...,POINT(-117.055176616996 32.5852793767876)
2,60730010002,60730010002,10002,073,001000,2,498,06,84853.0,69008.0,...,1436552.0,131594.0,69631.0,208272.0,16455832.0,33336356.0,181192.0,0106000020E61000000100000001030000000100000049...,MULTIPOLYGON(((-117.143778000325 32.7609579991...,POINT(-117.141940565517 32.7600381925693)
3,60730010003,60730010003,10003,073,001000,3,499,06,85964.0,67273.0,...,1749446.0,146427.0,81492.0,246413.0,18816988.0,41032398.0,201912.0,0106000020E61000000100000001030000000100000046...,MULTIPOLYGON(((-117.14501800108 32.75909699882...,POINT(-117.141939137067 32.7581694568747)
4,60730100031,60730100031,100031,073,010003,1,940,06,97939.0,86641.0,...,984980.0,105417.0,49493.0,157221.0,12200027.0,23428288.0,124310.0,0106000020E61000000100000001030000000100000070...,MULTIPOLYGON(((-117.036845001307 32.5837439990...,POINT(-117.040164080481 32.5786962650251)


In [19]:
# show rows with duplicated ctblockgrou
block_group_df[block_group_df.duplicated(subset=['ctblockgroup'], keep=False)]

,id,geo_id,ctblockgroup,countyfp,tractce,blkgrpce,ogc_fid,statefp,average_income,median_income,...,cs_restuarant,cs_school_food,cs_alcohol,cs_coffee_tea,cs_retail_goods,cs_housing,cs_pet_food,geom,geom_wkt,centroid_wkt


##### Entity 9: `Zone Location`

In [20]:
query = """
SELECT
    id,
    zone_name,
    imp_date,
    ordnum,
    shape_length,
    shape_area,
    legend,
    ST_Transform(geom, 4326) AS geom,
    ST_AsText(ST_Transform(geom, 4326)) AS geom_wkt,
    ST_AsText(ST_Centroid(ST_Transform(geom, 4326))) AS centroid_wkt
FROM sandag_layer_zoning_base_sd_new
"""

zone_location_df = query_db(query)
save_df_to_json(zone_location_df, "zone_location.json")
print(f"Rows: {zone_location_df.shape[0]}, Columns: {zone_location_df.shape[1]}")
zone_location_df.head()

Data saved to data/nodes/zone_location.json
Rows: 3677, Columns: 10


,id,zone_name,imp_date,ordnum,shape_length,shape_area,legend,geom,geom_wkt,centroid_wkt
0,1,AG-1-1,1141084800000,R-301263,281.423966,2.601150e+03,"Agricultural-General zone, use package 1, deve...",0106000020E6100000010000000103000000010000000E...,MULTIPOLYGON(((-117.12982648288 33.04642430605...,POINT(-117.129655123475 33.0465113079225)
1,6,AG-1-1,1141084800000,R-301263,3197.909381,2.041669e+05,"Agricultural-General zone, use package 1, deve...",0106000020E61000000100000001030000000100000085...,MULTIPOLYGON(((-117.037822960772 33.0743520276...,POINT(-117.036525139935 33.0757531247654)
2,7,AG-1-1,1141084800000,R-301263,85544.431621,6.369870e+07,"Agricultural-General zone, use package 1, deve...",0106000020E61000000100000001030000000100000040...,MULTIPOLYGON(((-117.106843080255 33.0731224473...,POINT(-117.104303504326 33.0609954579701)
3,8,AG-1-1,1141084800000,R-301263,17855.440197,7.212937e+06,"Agricultural-General zone, use package 1, deve...",0106000020E610000001000000010300000001000000C8...,MULTIPOLYGON(((-116.972290881461 33.0752706838...,POINT(-116.963679235586 33.0815738291832)
4,9,AG-1-1,1141084800000,R-301263,168.046707,1.087063e+03,"Agricultural-General zone, use package 1, deve...",0106000020E6100000010000000103000000010000000B...,MULTIPOLYGON(((-116.905763399499 33.0866176989...,POINT(-116.905850351155 33.086621495261)


##### Entity 10: `Zone Type`

In [21]:
query = """
SELECT
    DISTINCT(zone_name) as name,
    legend
FROM sandag_layer_zoning_base_sd_new
"""

zone_type_df = query_db(query)
zone_type_df["id"] = zone_type_df.index + 1
zone_type_df = zone_type_df[["id"]+[col for col in zone_type_df.columns if col != "id"]]
save_df_to_json(zone_type_df, "zone_type.json")
print(f"Rows: {zone_type_df.shape[0]}, Columns: {zone_type_df.shape[1]}")
zone_type_df.head()

Data saved to data/nodes/zone_type.json
Rows: 184, Columns: 3


,id,name,legend
0,1,OF-1-1,Open Space-Floodplain zone (floodplain areas)
1,2,CC-2-5,Other or Undefined Zone
2,3,CC-5-1,Commercial-Community zones (special areas)
3,4,CUPD-CT-2-3,Central Urbanized Planned District
4,5,LJPD-6A,La Jolla Planned District


##### Entity 11: `Sector`

In [22]:
sector_df = franchise_df[["category_sector"]].drop_duplicates().reset_index(drop=True)
sector_df["id"] = sector_df.index + 1
sector_df = sector_df[["id"]+[col for col in sector_df.columns if col != "id"]]
sector_df.rename(columns={"category_sector": "name"}, inplace=True)
save_df_to_json(sector_df, "sector.json")
print(f"Rows: {sector_df.shape[0]}, Columns: {sector_df.shape[1]}")
sector_df.head()

Data saved to data/nodes/sector.json
Rows: 12, Columns: 2


,id,name
0,1,Other Services
1,2,Entertainment & Recreation
2,3,Personal Services
3,4,Food & Beverage
4,5,Retail


##### Entity 12: `Subsector`

In [24]:
subsector_df = franchise_df[["category_subsector"]].drop_duplicates().reset_index(drop=True)
subsector_df["id"] = subsector_df.index + 1
subsector_df = subsector_df[["id"]+[col for col in subsector_df.columns if col != "id"]]
subsector_df.rename(columns={"category_subsector": "name"}, inplace=True)
save_df_to_json(subsector_df, "subsector.json")
print(f"Rows: {subsector_df.shape[0]}, Columns: {subsector_df.shape[1]}")
subsector_df.head()

Data saved to data/nodes/subsector.json
Rows: 45, Columns: 2


,id,name
0,1,Miscellaneous
1,2,Sports & Recreation
2,3,Health & Beauty
3,4,Specialty Food
4,5,Bars & Nightlife
